# 測試入門

## 學習目標

1. 理解為什麼自動化測試是軟體開發的必要環節
2. 使用 `unittest` 和 `pytest` 撰寫測試
3. 實踐 TDD（測試驅動開發）的 Red-Green-Refactor 流程
4. 使用 `doctest` 在文件字串中嵌入測試
5. 衡量測試覆蓋率並理解其意義

---

## 一、為什麼要寫測試？

| 情境 | 沒有測試 | 有測試 |
|:-----|:---------|:-------|
| 修 bug 後 | 祈禱沒弄壞其他功能 | 一鍵確認全部正常 |
| 重構程式碼 | 不敢動，怕改壞 | 放心改，測試會抓錯 |
| 多人協作 | 互相踩到對方的程式碼 | CI 自動跑測試，衝突立刻現形 |

---

## 二、unittest -- Python 內建測試框架

In [ ]:
import unittest

class Calculator:
    def add(self, a, b):
        return a + b
    def divide(self, a, b):
        if b == 0:
            raise ValueError("除數不能為零")
        return a / b

class TestCalculator(unittest.TestCase):
    def setUp(self):
        """每個測試方法前自動呼叫"""
        self.calc = Calculator()

    def test_add(self):
        self.assertEqual(self.calc.add(3, 5), 8)
        self.assertEqual(self.calc.add(-1, 1), 0)

    def test_divide(self):
        self.assertAlmostEqual(self.calc.divide(1, 3), 0.333, places=3)

    def test_divide_by_zero(self):
        with self.assertRaises(ValueError):
            self.calc.divide(6, 0)

    def tearDown(self):
        """每個測試方法後自動呼叫"""
        pass

### 常用斷言方法

| 方法 | 用途 |
|:-----|:-----|
| `assertEqual(a, b)` | `a == b` |
| `assertTrue(x)` / `assertFalse(x)` | 布林檢查 |
| `assertRaises(Error)` | 是否拋出指定例外 |
| `assertAlmostEqual(a, b)` | 浮點數近似比較 |
| `assertIn(a, b)` | `a in b` |

```bash
python -m unittest -v test_calculator.py
```

---

## 三、pytest -- 更現代的測試框架

```bash
pip install pytest
```

### 基本用法 -- 函式就能寫測試

In [ ]:
# test_calc_pytest.py
def test_add():
    calc = Calculator()
    assert calc.add(3, 5) == 8

def test_divide_by_zero():
    import pytest
    calc = Calculator()
    with pytest.raises(ValueError, match="除數不能為零"):
        calc.divide(6, 0)

### fixture -- 取代 setUp/tearDown

In [ ]:
import pytest

@pytest.fixture
def calc():
    return Calculator()

def test_add(calc):
    assert calc.add(3, 5) == 8

### parametrize -- 一次測試多組資料

In [ ]:
@pytest.mark.parametrize("a, b, expected", [
    (3, 5, 8), (-1, 1, 0), (-1, -1, -2), (0, 0, 0),
])
def test_add_many(calc, a, b, expected):
    assert calc.add(a, b) == expected

```bash
pytest -v                  # 詳細輸出
pytest -k "divide"         # 只跑名稱含 "divide" 的測試
```

---

## 四、TDD 實戰：密碼驗證器

流程：**Red**（寫測試，預期失敗）-> **Green**（最小實作）-> **Refactor**（改善結構）

### Red -- 先寫測試

In [ ]:
def test_too_short():
    assert PasswordValidator().validate("Ab1") is False

def test_no_digit():
    assert PasswordValidator().validate("NoDigitsHere") is False

def test_no_uppercase():
    assert PasswordValidator().validate("nouppercase123") is False

def test_valid():
    assert PasswordValidator().validate("GoodPass1") is True

### Green -- 最小實作

In [ ]:
class PasswordValidator:
    def validate(self, password: str) -> bool:
        if len(password) < 8:
            return False
        if not any(c.isdigit() for c in password):
            return False
        if not any(c.isupper() for c in password):
            return False
        return True

### Refactor -- 改善結構

In [ ]:
class PasswordValidator:
    MIN_LENGTH = 8

    def validate(self, password: str) -> bool:
        return all([
            len(password) >= self.MIN_LENGTH,
            any(c.isdigit() for c in password),
            any(c.isupper() for c in password),
        ])

再跑 `pytest -v`，確認測試仍全部通過。

---

## 五、doctest -- 文件即測試

In [ ]:
def factorial(n):
    """
    計算 n 的階乘。

    >>> factorial(0)
    1
    >>> factorial(5)
    120
    >>> factorial(-1)
    Traceback (most recent call last):
        ...
    ValueError: n 必須為非負整數
    """
    if n < 0:
        raise ValueError("n 必須為非負整數")
    return 1 if n == 0 else n * factorial(n - 1)

```bash
python -m doctest my_module.py -v
```

> 適合簡單工具函式。複雜邏輯請用 pytest。

---

## 六、執行測試與閱讀輸出

```
$ pytest -v
test_calculator.py::test_add PASSED
test_calculator.py::test_divide PASSED
test_calculator.py::test_divide_by_zero PASSED
========================= 3 passed in 0.02s =========================
```

- **PASSED**：通過
- **FAILED**：斷言失敗（pytest 自動顯示實際值 vs 預期值）
- **ERROR**：測試程式碼本身出錯

---

## 七、測試覆蓋率

```bash
pip install pytest-cov
pytest --cov=my_package tests/          # 終端報告
pytest --cov=my_package --cov-report=html tests/  # HTML 報告
```

```
Name                    Stmts   Miss  Cover
-------------------------------------------
my_package/calculator.py   15      2    87%
my_package/validator.py    10      2    80%
TOTAL                      25      4    84%
```

- **80% 是常見的合理目標**，涵蓋主要路徑與邊界情況
- 100% 不一定值得追求 -- 重要的是測試有意義的行為

---

## 八、unittest vs pytest 比較

| 特性 | unittest | pytest |
|:-----|:---------|:-------|
| 安裝 | 內建 | `pip install pytest` |
| 測試寫法 | 繼承 `TestCase` | 函式即可 |
| 斷言 | `self.assertEqual(a, b)` | `assert a == b` |
| 共用設定 | `setUp` / `tearDown` | `@pytest.fixture` |
| 參數化 | 需要額外套件 | `@pytest.mark.parametrize` |
| 失敗訊息 | 基本 | 自動差異比較 |
| 適用場景 | 維護舊專案 | 新專案首選 |

> **建議**：新專案用 pytest。pytest 也能直接執行 unittest 的測試案例。

---

[<< 回到課程總覽](../00_course_outline.md)